In [ ]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Welcome! I set up a pre-processing python file so that everyone could more easily see, understand, and work on the data for our project. I will be adding more code to this eventually, but for now this is a decent start. You should be able to run all of this too (I think), just make sure you have your google drive mounted🙂

In [ ]:
#All data uploaded neatly into data frames to mess around with
df_song = pd.read_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550/songs.csv")
df_artist = pd.read_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/spotify_550/artists.csv")
df_billboard = pd.read_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/billboard/charts.csv")
df_grammy = pd.read_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/grammy/grammy_winners.csv")

Updating the Grammys Dataset!

In [ ]:
#old version of df_grammy
#df_grammy.head(20)

In [ ]:
# extract from song_or_album first
type_from_title = (
    df_grammy["song_or_album"]
    .str.extract(r"\((Album|Single)\)", expand=False)
    .str.lower()
    .replace({"single": "song"})
)

# extract from award column if first fails
type_from_award = (
    df_grammy["award"]
    .str.extract(r"(Album|Song)", expand=False)
    .str.lower()
)

# combine them
df_grammy["type"] = type_from_title.fillna(type_from_award).fillna("song")

#extract the name (remove the parentheses part)
df_grammy["name"] = df_grammy["song_or_album"].str.replace(r"\s*\((Album|Single)\)", "", regex=True)

#drop the original column called "song_or_album"
df_grammy = df_grammy.drop(columns=["song_or_album"])

In [ ]:
# #extract album/single into type column by album/song
# df_grammy["type"] = (
#     df_grammy["song_or_album"]
#     .str.extract(r"\((Album|Single)\)", expand=False)
#     .str.lower()
#     .replace({"single": "song"})
#     .fillna("song")     # default if no tag exists
# )

# #extract the name (remove the parentheses part)
# df_grammy["name"] = df_grammy["song_or_album"].str.replace(r"\s*\((Album|Single)\)", "", regex=True)

# #drop the original column called "song_or_album"
# df_grammy = df_grammy.drop(columns=["song_or_album"])

In [ ]:
#print(df_grammy.head(30))

In [ ]:
#fill in weird NaN values with pre-existing data, and then replace NaN with "Not Available"
df_grammy["artist"] = df_grammy["artist"].fillna(
    df_grammy.groupby("name")["artist"].transform("first")
)
df_grammy["artist"] = df_grammy["artist"].fillna("Not Available")

#add id column
df_grammy.insert(0, "id", range(1, len(df_grammy) + 1))

#drop url column
df_grammy = df_grammy.drop(columns=["url"])
#drop producers and artist_link
df_grammy = df_grammy.drop(columns=["producers"])
df_grammy = df_grammy.drop(columns=["artist_link"])

In [ ]:
df_grammy.head(30)

In [ ]:
df_grammy.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/grammy_cleaned/grammy_cleaned.csv", index=False)

Now to update Billboard 100

In [ ]:
#add id column
df_billboard.insert(0, "id", range(1, len(df_billboard) + 1))

#df_billboard.to_csv("billboard_updated.csv", index=False)
df_billboard.to_csv("/content/drive/MyDrive/CIS5500 Group Project/Coding/Data/billboard_cleaned/billboard_cleaned.csv", index=False)

In [ ]:
#some little functions I made for testing out things and getting a quick visual of the data
def simple_stats(column_names, df):
    """This function outputs simple stats for a given dataframe and first 15 items from a list of column names
        Inputs:
        column_names - a list of strings of column names from a df
        df - the dataframe we are analyzing, should already be read in and saved as a pandas df
    """
    print("DF Shape:", df.shape)
    print("Total Rows:", df.shape[0])
    print("Total Columns:", df.shape[1])
    print("All Column names:", list(df.columns))
    print("Selected Columns Output:")

    print(df[column_names].head(30))

def merge_inner_join(df1, df2, left_on_name, right_on_name):
  """This function is just for easily testing our merging different data frames with inner join to see how it goes.
  """
  merged_df = pd.merge(df1, df2, left_on=left_on_name, right_on=right_on_name, how="inner")
  print("DF Shape:", merged_df.shape)
  print("Total Rows:", merged_df.shape[0])
  print("Total Columns:", merged_df.shape[1])
  print(merged_df.head(15))

In [ ]:
#Simple Stats for Grammys
simple_stats(["artist", "award", "name"], df_grammy)

In [ ]:
#Quick View of Billboards
simple_stats(["date", "song", "artist"], df_billboard)